# Batch density fits for phase-separated thermalized states

This notebook discovers every V3 thermalization state whose saved voxel classification is phase separated, fits its gas/liquid/interface voxel histogram with the **same pooled-tail routine used for cavitation states**, and writes a resumable CSV.

For each state, it reports the fitted gas density, fitted liquid density, and liquid/gas density ratio. Standard errors and 95% confidence intervals are propagated from the complete five-parameter fit covariance, including parameter correlations.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from md_Helpers.paths import MASTER_CSVS_V3_ROOT, THERMALIZED_STATES_V3_ROOT
from md_Helpers.phase_density import (
    analyze_phase_separated_thermalized_states,
    discover_phase_separated_thermalized_states,
    plot_density_ratio_with_uncertainty,
    select_phase_density_results,
)

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

## 1. Batch settings

These are the cavitation voxel-fit defaults: five tail frames, five-frame spacing, final half of the trajectory, 40 interface quadrature points, and a 50/50 interface allocation. `voxel_nbins=None` applies the repository's `ncell`-dependent Seitz resolution rule.

In [ ]:
THERMALIZED_ROOT = Path(THERMALIZED_STATES_V3_ROOT)
OUTPUT_CSV = Path(MASTER_CSVS_V3_ROOT) / "thermalized_phase_density_fits.csv"

VOXEL_NBINS = None
NFRAMES = 5
FRAME_SKIP = 5
TAIL_FRACTION = 0.5
INTERFACE_VOID_FRACTION = 0.5
INTERFACE_POINTS = 40
MAX_FIT_ITERATIONS = 500

print(f"Thermalized root: {THERMALIZED_ROOT}")
print(f"Resumable output: {OUTPUT_CSV}")

## 2. Discover classified phase-separated states

This reads existing HDF5 classification metadata. It does not reclassify states or run simulations.

In [ ]:
phase_states = discover_phase_separated_thermalized_states(
    root=THERMALIZED_ROOT
)
print(f"Discovered {len(phase_states)} phase-separated thermalized states")
display(phase_states)

## 3. Fit every discovered state

The CSV is checkpointed after every state. With `resume=True`, completed fits with identical analysis settings are reused on the next run. Failed fits are attempted again.

Legacy thermalization `randomization.gsd` files often contain one frame. The exact cavitation fitter is still used, but `frames_available` and `frames_fitted` will be 1 for those files.

In [ ]:
results = analyze_phase_separated_thermalized_states(
    states=phase_states,
    output_path=OUTPUT_CSV,
    voxel_nbins=VOXEL_NBINS,
    nframes=NFRAMES,
    skip=FRAME_SKIP,
    tail_fraction=TAIL_FRACTION,
    interface_void_fraction=INTERFACE_VOID_FRACTION,
    interface_points=INTERFACE_POINTS,
    max_iterations=MAX_FIT_ITERATIONS,
    resume=True,
)

print(results["status"].value_counts(dropna=False))
print(f"Saved {len(results)} rows to {OUTPUT_CSV}")

In [ ]:
quality_columns = [
    "n_fcc_cells", "target_rho", "kT", "seed", "status",
    "frames_available", "frames_fitted", "fit_success",
    "gas_density", "liquid_density",
    "liquid_to_gas_density_ratio", "error",
]
display(results[[column for column in quality_columns if column in results]])

failed = results[results["status"].ne("completed")]
if not failed.empty:
    print(f"WARNING: {len(failed)} fits did not complete. Inspect their error column.")

completed = results[results["status"].eq("completed")]
for column in ["n_fcc_cells", "kT", "target_rho", "seed"]:
    values = sorted(completed[column].dropna().unique())
    print(f"Available {column}: {values}")

## 4. Select the states to inspect

Each selector accepts a list, one scalar value, or `None` for all values. For example, use `SELECT_NCELLS = [20, 30]` or `SELECT_TEMPERATURES = [0.7, 0.8]`.

In [ ]:
SELECT_NCELLS = [30]
SELECT_TEMPERATURES = None
SELECT_TARGET_DENSITIES = None
SELECT_SEEDS = None
RATIO_ERROR_BARS = "se"  # Choose "se" or "ci95".

selected = select_phase_density_results(
    results,
    ncells=SELECT_NCELLS,
    temperatures=SELECT_TEMPERATURES,
    target_densities=SELECT_TARGET_DENSITIES,
    seeds=SELECT_SEEDS,
)

selected_columns = [
    "n_fcc_cells", "target_rho", "actual_rho", "kT", "seed",
    "gas_density", "gas_density_se",
    "liquid_density", "liquid_density_se",
    "liquid_to_gas_density_ratio",
    "liquid_to_gas_density_ratio_se",
    "liquid_to_gas_density_ratio_ci95_low",
    "liquid_to_gas_density_ratio_ci95_high",
    "frames_fitted", "state_path",
]
print(f"Selected {len(selected)} completed fits")
display(selected[[column for column in selected_columns if column in selected]])

## 5. Filtered gas and liquid density plots

These use the propagated one-standard-error uncertainties. Curves are separated by `ncell` and target density when necessary.

In [ ]:
if selected.empty:
    print("No completed fits match the current filters.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    group_columns = ["n_fcc_cells", "target_rho"]
    for (ncell, rho), group in selected.groupby(group_columns, dropna=False):
        group = group.sort_values("kT")
        label = f"ncell={ncell:g}, rho={rho:g}"
        axes[0].errorbar(
            group["kT"], group["gas_density"],
            yerr=group["gas_density_se"], marker="o", capsize=3, label=label,
        )
        axes[1].errorbar(
            group["kT"], group["liquid_density"],
            yerr=group["liquid_density_se"], marker="o", capsize=3, label=label,
        )

    axes[0].set(xlabel="kT", ylabel="Gas density", title="Fitted gas density")
    axes[1].set(xlabel="kT", ylabel="Liquid density", title="Fitted liquid density")
    for axis in axes:
        axis.grid(alpha=0.3)
        axis.legend()
    plt.show()

## 6. Liquid/gas density ratio with uncertainty

When the filtered data contain one `ncell`, the x-axis is `kT` and the y-axis is the fitted liquid/gas density ratio. Error bars are one propagated standard error. Different target densities are shown as separate series.

In [ ]:
if selected.empty:
    print("No completed fits match the current filters.")
else:
    ax = plot_density_ratio_with_uncertainty(
        selected, uncertainty=RATIO_ERROR_BARS
    )
    # ax.set_yscale("log")  # Optional when ratios span a large range.
    # ax.figure.savefig("thermalized_density_ratio.png", dpi=300, bbox_inches="tight")
    plt.show()

## Uncertainty interpretation

The reported uncertainty is the local fit uncertainty from the likelihood Hessian. It includes covariance between fit parameters, but it does not include model-selection uncertainty, sensitivity to voxel resolution, or correlations between neighboring voxels. For legacy single-frame thermalized states, it also cannot measure frame-to-frame variation. Use `frames_fitted` when interpreting or filtering the results.